# Athletes with the Most Lange Laufnacht Meetings

This notebook develops a useful definition of participation step by step. It starts with raw result rows, then removes non-starters and non-finishers, and finally counts at most one meeting per athlete and year. Athlete names are compared as sets of whitespace-separated words because the 2021 and 2022 sources switched the order of given name and surname.

The final metric is: **number of distinct event years in which an athlete finished at least one race**. The meeting did not take place in 2020, so that year is absent rather than treated as a missed participation.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "lln_stats").exists():
            return path
    raise FileNotFoundError("Could not find the lln-stats repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RESULTS_PATH = REPO_ROOT / "features" / "results.parquet"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

df = pd.read_parquet(RESULTS_PATH)


def athlete_name_key(name: str) -> tuple[str, ...]:
    """Create an order-independent, deterministic key from the words in a name."""
    return tuple(sorted(set(name.casefold().split())))


df["athlete_key"] = df["athlete_name"].map(athlete_name_key)

# Prefer the repository's usual surname-first spelling for display when it exists.
display_names = (
    df.assign(switched_name_year=df["event_year"].isin([2021, 2022]))
    .sort_values(["switched_name_year", "event_year"])
    .drop_duplicates("athlete_key")
    .set_index("athlete_key")["athlete_name"]
)

{
    "rows": len(df),
    "distinct_raw_names": df["athlete_name"].nunique(),
    "matched_athletes": df["athlete_key"].nunique(),
    "meeting_years": sorted(df["event_year"].unique().tolist()),
    "source": str(RESULTS_PATH.relative_to(REPO_ROOT)),
}

## 1. Naive count: result rows

The simplest query counts every row carrying an athlete's name. This is easy, but it treats DNS and DNF records as participation and can count multiple races at one meeting more than once.

In [ ]:
naive_row_counts = (
    df.groupby("athlete_key", as_index=False)
    .agg(result_rows=("event_year", "size"))
    .assign(athlete_name=lambda x: x["athlete_key"].map(display_names))
    .sort_values(["result_rows", "athlete_name"], ascending=[False, True])
)

naive_row_counts.head(20)

## 2. Distinct meeting years, regardless of result

Counting distinct years fixes the multiple-races problem. It still includes athletes who were listed but did not finish. The status table shows how large that distinction is.

In [ ]:
status_summary = (
    df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="result_rows")
)

listed_meetings = (
    df.groupby("athlete_key", as_index=False)
    .agg(listed_meetings=("event_year", "nunique"))
    .assign(athlete_name=lambda x: x["athlete_key"].map(display_names))
    .sort_values(["listed_meetings", "athlete_name"], ascending=[False, True])
)

status_summary, listed_meetings.head(20)

## 3. Keep only finished races

A recorded finish is represented by `status == "finished"`. Requiring it gives participation a concrete sporting meaning. Before deduplicating, inspect athlete-years with multiple finished rows so the need for the final step remains visible.

In [ ]:
finished = df[df["status"].eq("finished")].copy()

finished_rows_per_athlete_year = (
    finished.groupby(["athlete_key", "event_year"], as_index=False)
    .agg(finished_races=("event", "size"), events=("event", lambda s: ", ".join(sorted(set(s)))))
    .assign(athlete_name=lambda x: x["athlete_key"].map(display_names))
)

multiple_finishes_at_one_meeting = finished_rows_per_athlete_year[
    finished_rows_per_athlete_year["finished_races"].gt(1)
].sort_values(["finished_races", "event_year"], ascending=[False, True])

multiple_finishes_at_one_meeting

## 4. Final definition: one finished participation per year

Drop duplicate athlete-year combinations after filtering to finishers. The resulting table has exactly one row per qualifying meeting participation.

In [ ]:
finished_participations = (
    finished.sort_values(["athlete_name", "event_year", "time_seconds"])
    .drop_duplicates(["athlete_key", "event_year"])
)

assert not finished_participations.duplicated(["athlete_key", "event_year"]).any()

leaderboard = (
    finished_participations.groupby("athlete_key", as_index=False)
    .agg(
        meetings_finished=("event_year", "size"),
        first_finish=("event_year", "min"),
        latest_finish=("event_year", "max"),
        years=("event_year", lambda s: ", ".join(map(str, sorted(s)))),
    )
    .assign(athlete_name=lambda x: x["athlete_key"].map(display_names))
    .sort_values(
        ["meetings_finished", "latest_finish", "athlete_name"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
leaderboard.index = leaderboard["meetings_finished"].rank(method="dense", ascending=False).astype(int)
leaderboard.index.name = "rank"

leaderboard.head(30)

## Compare the definitions

Putting the three metrics side by side makes the effect of each business rule explicit.

In [ ]:
comparison = (
    naive_row_counts.drop(columns="athlete_name")
    .merge(listed_meetings.drop(columns="athlete_name"), on="athlete_key", how="outer")
    .merge(
        leaderboard.reset_index()[["athlete_key", "athlete_name", "meetings_finished"]],
        on="athlete_key",
        how="left",
    )
    .fillna({"meetings_finished": 0})
)
comparison["meetings_finished"] = comparison["meetings_finished"].astype(int)
comparison["listed_but_not_finished"] = (
    comparison["listed_meetings"] - comparison["meetings_finished"]
)

comparison.sort_values(
    ["meetings_finished", "listed_meetings", "result_rows", "athlete_name"],
    ascending=[False, False, False, True],
).head(30)

## Identity quality checks

This analysis identifies a person by the set of case-insensitive, whitespace-separated words in `athlete_name`. This joins names whose word order changed in 2021 and 2022, but equal name-word sets can theoretically refer to different people and spelling changes can still split one person into multiple records. The first table shows keys that merged multiple raw name variants; the second flags conflicting birth years or genders for review.

In [ ]:
identity_checks = (
    finished.groupby("athlete_key", as_index=False)
    .agg(
        name_variants=("athlete_name", lambda s: sorted(s.unique().tolist())),
        birth_year_values=("year_of_birth", lambda s: sorted(s.dropna().unique().tolist())),
        gender_values=("gender", lambda s: sorted(s.dropna().unique().tolist())),
    )
)
matched_name_variants = identity_checks[identity_checks["name_variants"].str.len().gt(1)]
identity_conflicts = identity_checks[
    identity_checks["birth_year_values"].str.len().gt(1)
    | identity_checks["gender_values"].str.len().gt(1)
]

matched_name_variants, identity_conflicts

## Drill into a leading athlete

Change `ATHLETE_QUERY` to inspect every recorded result for a person, including non-finishes. This is useful for validating a leaderboard entry and understanding which events they ran.

In [ ]:
ATHLETE_QUERY = leaderboard.iloc[0]["athlete_name"]
ATHLETE_KEY = athlete_name_key(ATHLETE_QUERY)

athlete_history = df[df["athlete_key"].map(lambda key: key == ATHLETE_KEY)][
    [
        "event_year",
        "athlete_name",
        "event",
        "status",
        "result_raw",
        "club",
        "year_of_birth",
    ]
].sort_values(["event_year", "event"])

athlete_history